# Capacitated facility location

In this notebook, you will implement an integer program for the *capacitated facility location* problem. We allow either a small or a large depot to be opened at each location (but not both). You can find more about the problem in Section 2.6 of the lecture notes. The notebook
[facility.ipnyb](https://colab.research.google.com/github/henningbruhn/opt1/blob/main/facility_en.ipynb)
will also be useful to you.

\begin{align}
	\min\quad & \sum_{i\in\mathcal F}y_i^kf_i^k+y_i^gf_i^g
+\sum_{i\in\mathcal F,j\in\mathcal D}x_{ij}c_{ij},
& y^{k},y^g\in\mathbb R^{\mathcal F},\, x\in\mathbb R^{\mathcal F\times\mathcal D}\\
	\text{subject to}\quad & \sum_{j\in\mathcal D} x_{ij}\leq q^ky_i^k+q^gy_i^g
& \text{for all }i\in\mathcal F\\
& y_i^k+y_i^g\leq 1 & \text{for all }i\in\mathcal F\\
	& \sum_{i\in\mathcal F}x_{ij}\geq 1 & \text{for all }j\in\mathcal D\\
	& x_{ij}\in\{0,1\},\,y_i^k,y_i^g\in\{0,1\} & \text{for all }i\in\mathcal F,\,j\in\mathcal D
\end{align}




The <code>mip</code> package is used. Documentation can be found [here.](https://python-mip.readthedocs.io/en/latest/)
If you are using Google Colab or a Linux system (or Mac?), the code below should ensure that <code>mip</code> is installed if necessary. Otherwise, you may need to install <code>mip</code> manually. Since the last time I touched a Windows system was 25 years ago, you better ask an AI.

In [1]:
# only install mip if not already present
try:
    import mip
except:
    !pip install mip

import mip  ## Library for linear programs and MIPs
import random
import math
import matplotlib.pyplot as plt
import time

## Generation of a random instance

As always, we need an example instance, which we generate randomly. We place facility locations and customers randomly in the plane. For the connection/operating costs, we take the standard Euclidean distance in the plane. We choose the construction costs randomly. The instance is stored in a [dictionary](https://docs.python.org/3/tutorial/datastructures.html#dictionaries). This is a data structure that can hold a wide variety of data via keywords. In particular, the following information is contained in the dictionary <code>inst</code>:
* <code>inst["facilities"]</code>: a list with the plane coordinates of the facility locations for depots
* <code>inst["customers"]</code>: a list with the plane coordinates of the customers
* <code>inst["opening_costs_small"]</code>: a list with the construction costs for small depots
* <code>inst["opening_costs_large"]</code>: a list with the construction costs for large depots
* <code>inst["capacity_small"]</code>: the capacity of a small depot (same for all small depots)
* <code>inst["capacity_large"]</code>: the capacity of a large depot (same for all large depots)
* <code>inst["num_facilities"]</code>: number of facilities
* <code>inst["num_customers"]</code>: number of customers

In [2]:
### You do not need to understand this code. The only important thing is that the instance is stored in the variable inst.
def rnd_instance(num_facilities=30,num_customers=1000):
    random.seed(42)
    inst={}
    inst["facilities"]=[(random.random(),random.random()) for _ in range(num_facilities)]
    inst["customers"]=[(random.random(),random.random()) for _ in range(num_customers)]
    inst["opening_costs_small"]=[random.randint(7,14) for _ in range(num_facilities)]  #[10]*num_facilities
    inst["opening_costs_large"]=[random.randint(20,34) for _ in range(num_facilities)]
    inst["capacity_small"]=math.floor(0.5*num_customers/num_facilities)
    inst["capacity_large"]=round(3*num_customers/num_facilities)
    inst["num_facilities"]=num_facilities
    inst["num_customers"]=num_customers
    return inst

def distance(facility,customer):
    px,py=facility
    qx,qy=customer
    return math.sqrt((px-qx)**2+(py-qy)**2)

# The actual instance:
inst=rnd_instance()

How do we access the instance data? Like this:

In [3]:
inst["capacity_small"]

16

### Task: facilities
Implement the above integer program. Use the code below to print the costs and properties of the solution as well as the computation time. For this, please create the following variables in the integer program:
* <code>opening_costs</code> for the construction costs
* <code>connection_costs</code> for the operating costs
* <code>y_k</code> one binary variable per facility to indicate whether a small depot is opened there
* <code>y_g</code> one binary variable per facility to indicate whether a large depot is opened there

The easiest way to do this task is to adapt the code from the notebook
[facility.ipnyb](https://colab.research.google.com/github/henningbruhn/opt1/blob/main/facility_en.ipynb).

### Solution



In [4]:
model=mip.Model()
opening_costs=model.add_var()
connection_costs=model.add_var()
facilities = inst["facilities"]
customers = inst["customers"]
num_facilities = inst["num_facilities"]
y_k = [model.add_var(var_type=mip.BINARY) for _ in range(num_facilities)]
y_g = [model.add_var(var_type=mip.BINARY) for _ in range(num_facilities)]
x=[[model.add_var(var_type=mip.BINARY) for _ in range(inst["num_customers"])] for _ in range(inst["num_facilities"])]

opening_costs = mip.xsum(inst["opening_costs_small"][i] * y_k[i] + inst["opening_costs_large"][i] * y_g[i] for i in range(num_facilities))
connection_costs = mip.xsum(x[i][j] * distance(facilities[i], customers[j]) for i in range(num_facilities) for j in range(inst["num_customers"]))

# Each Customer must be connected to at least 1 facility
for j in range(inst["num_customers"]):
    model+=mip.xsum(x[i][j] for i in range(num_facilities)) >= 1

for i in range(num_facilities):
    model+=mip.xsum(x[i][j] for j in range(inst["num_customers"])) <= inst["capacity_small"]*y_k[i]+inst["capacity_large"]*y_g[i]

# Only 1 Depot per Location
for i in range(num_facilities):
    model+=y_k[i] + y_g[i] <= 1

model.objective = mip.minimize(opening_costs + connection_costs)

We start the optimization and measure the computation time.

In [5]:
start=time.time() # Code for time measurement
model.optimize()
end=time.time()
total_time=end-start

total_costs=model.objective_value
num_small_facilities=sum([y_k[i].x for i in range(num_facilities)])
num_large_facilities=sum([y_g[i].x for i in range(num_facilities)])

print("Total costs: {}".format(round(total_costs,1)))
print("  of which operating costs: {}".format(round(connection_costs.x,1)))
print("  of which construction costs: {}".format(round(opening_costs.x,1)))
print("Number of large / small depots: {} / {}".format(num_large_facilities,num_small_facilities))
print("Computation time for optimization: {}s".format(round(total_time,1)))


▶ Problem loading

  |Rows| = 1060   |Cols| = 30062   |NZ| = 60120
  Variables: 30060 binary, 0 integer, 2 continuous

  Coefficient ranges:
    Matrix  ∈ [        1, 100]   κ ≈    100
    Cost    ∈ [0.0035005,  34]   κ ≈ 9712.9
    Bounds  ∈ [        1,   1]   κ ≈      1
    RHS     ∈ [        1,   1]   κ ≈      1

✔ Problem loaded — 1060 rows, 30062 cols, 60120 NZ, 30060 integer

▶ Non-default parameters

  allowableGap  1e-12 → 1e-10
  ratioGap  0 → 0.0001
  lpTimeFreq  5 → 0

▶ Root LP relaxation

  Bound tightening: 0 bounds tightened (0 fixed) in 0.001 s.
  Clique strengthening: no changes (0.02s)
  Coefficient strengthening: no changes (0.00s)
  Row reductions: no changes (0.00s)
  LP auto: recommended primal_sprint

▶ Preprocessing


  Processed model: 1060 rows, 30060 cols, 60120 NZ
                   30060 integer (30060 binary)

  Conflict graph: built in 0.02s, density 0.002%
  Clique strengthening: extended 0, dominated 0

✔ Preprocessing complete — Time: 0.135s

▶ Root n

### Task: Relaxation
Implement the integer program once more, but this time with real-valued variables. That is, replace each binary variable with one that can take values in $[0,1]$. This gives you a standard linear program, often called the (linear) *relaxation* of the original program. Run the optimization and compare the optimal costs and computation time with the integer program above. What do you observe? Answer **briefly** in a *markdown* cell. (For robust statements about the computation time, one would have to solve both the integer program and the relaxation repeatedly and average the computation times. We will spare ourselves that.)

### Solution

In [6]:
model=mip.Model()
opening_costs=model.add_var()
connection_costs=model.add_var()
num_facilities = inst["num_facilities"]
y_k = [model.add_var(var_type=mip.CONTINUOUS, ub=1) for i in range(num_facilities)]
y_g = [model.add_var(var_type=mip.CONTINUOUS, ub=1) for i in range(num_facilities)]
x=[[model.add_var(var_type=mip.CONTINUOUS) for _ in range(inst["num_customers"])] for _ in range(inst["num_customers"])]

# Each Customer must be connected to at least 1 facility
for j in range(inst["num_customers"]):
    model+=mip.xsum(x[i][j] for i in range(num_facilities)) >= 1

for i in range(num_facilities):
    model+=mip.xsum(x[i][j] for j in range(num_facilities)) <= inst["opening_costs_small"][i] * y_k[i] + inst["opening_costs_large"][i] * y_g[i]

# Only 1 Depot per Location
for i in range(num_facilities):
    model+=y_k[i] + y_g[i] <= 1

model.objective = mip.minimize(mip.xsum(inst["opening_costs_small"][i] * y_k + inst["opening_costs_large"][i] * y_g) + mip.xsum(mip.xsum(x[i][j] * distance(inst["facilities"][i],inst["customers"][j]) for i in range(num_facilities)) for j in range(num_facilities)))

Identical code as above.

In [7]:
start=time.time()
model.optimize()
end=time.time()
total_time=end-start

total_costs=model.objective_value
num_small_facilities=sum([y_k[i].x for i in range(num_facilities)])
num_large_facilities=sum([y_g[i].x for i in range(num_facilities)])

print("Total costs: {}".format(round(total_costs,1)))
print("  of which operating costs: {}".format(round(connection_costs.x,1)))
print("  of which construction costs: {}".format(round(opening_costs.x,1)))
print("Number of large / small depots: {} / {}".format(num_large_facilities,num_small_facilities))
print("Computation time for optimization: {}s".format(round(total_time,1)))

CBC devel (git:146ce89) — COIN-OR Branch and Cut
Total time (Wallclock seconds): 0.000262022   (CPU seconds):             0.000256
  LP auto: recommended dual_pesteep_psi1_pertv61
0  Obj 0 Primal inf 13.275676 (30)
Optimal - objective value 27.234036
Optimal objective 27.23403646 - 105 iterations time 0.092, Presolve 0.09
Total costs: 27.2
  of which operating costs: 0.0
  of which construction costs: 0.0
Number of large / small depots: 0.08823529411764708 / 2.0805860805860803
Computation time for optimization: 0.3s


**What do you observe?**